# Catalog Creation

Use this notebook to inspect the LWI Region 3 source GeoJSON files, generate the StormHub catalog config, create the base STAC catalog, and then serve the generated catalog with `stormhub-server`.

In [ ]:
from pathlib import Path
import json

import geopandas as gpd

from stormhub.logger import initialize_logger
from stormhub.met.storm_catalog import new_catalog

# If this cell fails, adjust REPO_ROOT to point at your local stormhub checkout.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data" / "lwi-region3"
CATALOG_ROOT = REPO_ROOT / "catalogs"
CATALOG_ID = "lwi-region3"
CATALOG_DIR = CATALOG_ROOT / CATALOG_ID
CONFIG_PATH = CATALOG_DIR / "lwi-r3-config.json"

WATERSHED_PATH = DATA_DIR / "lwi_r3_huc12_domain.json"
TRANSPOSITION_REGION_PATH = DATA_DIR / "lwi_r3_transposition_domain.json"

DATA_DIR, CATALOG_DIR, CONFIG_PATH

## Inspect Source Geometries

StormHub expects each configured geometry to resolve to a single polygon after CRS handling. These checks make the source CRS, geometry type, feature count, and bounds visible before catalog creation.

In [ ]:
def describe_geojson(path: Path) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    display({
        "path": str(path.relative_to(REPO_ROOT)),
        "features": len(gdf),
        "crs": str(gdf.crs),
        "geometry_types": sorted(gdf.geometry.geom_type.unique().tolist()),
        "bounds": tuple(round(v, 6) for v in gdf.total_bounds),
    })
    return gdf

watershed_gdf = describe_geojson(WATERSHED_PATH)
transposition_gdf = describe_geojson(TRANSPOSITION_REGION_PATH)

In [ ]:
ax = transposition_gdf.boundary.plot(figsize=(8, 8), color="tab:orange", linewidth=2)
watershed_gdf.boundary.plot(ax=ax, color="tab:blue", linewidth=1)
ax.set_title("LWI Region 3 Watershed and Transposition Region")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude");

## Write StormHub Config

This keeps source files under `data/lwi-region3` and generated STAC catalog files under `catalogs/lwi-region3`.

In [ ]:
config = {
    "watershed": {
        "id": "lwi-r3-huc12-domain",
        "geometry_file": str(WATERSHED_PATH),
        "description": "LWI Region 3 HUC12 watershed domain",
    },
    "transposition_region": {
        "id": "lwi-r3-transposition-domain",
        "geometry_file": str(TRANSPOSITION_REGION_PATH),
        "description": "LWI Region 3 transposition domain",
    },
}

CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(CONFIG_PATH)
config

## Create Base STAC Catalog

This creates `catalogs/lwi-region3/catalog.json` and the generated `hydro_domains` STAC items. It may access the AORC S3 dataset while creating the valid transposition-region item.

In [ ]:
initialize_logger()

catalog = new_catalog(
    catalog_id=CATALOG_ID,
    config_file=str(CONFIG_PATH),
    local_directory=str(CATALOG_ROOT),
    catalog_description="LWI Region 3 storm catalog",
)

catalog.spm.catalog_file

In [ ]:
sorted(p.relative_to(CATALOG_DIR) for p in CATALOG_DIR.rglob("*.json"))

## Serve The Catalog

Run this from a PowerShell terminal with the `stormhub` environment active:

```powershell
stormhub-server .\catalogs\lwi-region3 127.0.0.1 5000
```

Then open `http://localhost:5000/catalog.json` or use the STAC Browser link shown by the directory listing.